# 03 — Data Profiling Review

**Goal:** Identify what's *structurally notable* about each raw table — not broken
(that's 02's job), but worth a deliberate decision before dbt staging models are written.

This notebook reads `audit.profiling_reports` (already computed by the pipeline for every
table in `raw`) to answer:

- Which tables have constant columns (single distinct value) — candidates to drop?
- Which tables have columns with outliers — worth a closer look, not necessarily wrong?
- Which tables have highly correlated column pairs — possible redundant/derivable columns?
- For flagged tables, what do the actual columns look like (via `report_json`)?

This notebook does **not** query `raw.*` tables directly — that's 04. Here we only read
the pre-computed profiling metadata, including drilling into `report_json` for column-level detail
that the summary counts alone don't show.

Output of this notebook: a **shortlist of columns to drop or double-check per table**, saved at the bottom.

In [14]:
from _bootstrap import project_root
from src.audit.run_management import get_latest_run_id, list_run_ids

import json
import polars as pl

# Show all columns and full-width string values in every cell below —
# otherwise polars truncates wide tables with "…" and cuts long strings like file_path/error
pl.Config.set_tbl_cols(-1)          # show all columns, no collapsing
pl.Config.set_tbl_width_chars(200)  # widen the rendered table
pl.Config.set_fmt_str_lengths(120)  # don't truncate long strings (e.g. file_path, error messages)
pl.Config.set_tbl_rows(50)          # show more rows before truncating vertically

# Connect to the project's DuckDB instance
from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(True)
print("Connected")

Connected


## 1. Get the latest run and load `audit.profiling_reports`

Use `get_latest_run_id()` (from `src.audit.run_management`) to find the most recent
`raw` run directly, rather than inferring it from `generated_at` ourselves. Filter to
just that run at the SQL level.

We still confirm afterward that this run covers all 369 raw tables — if a future run
only profiled a subset (like 02 saw for `dq_reports`, where some tables only appear
in an earlier run), we don't want to silently lose those tables.

In [15]:
latest_run_id = get_latest_run_id(conn, "audit.profiling_reports", layer="raw")
print(f"Latest profiling run_id: {latest_run_id}")

latest_profiling = conn.execute(
    """
    SELECT run_id, layer, table_name, generated_at, report_version,
           total_rows, total_columns, numeric_columns, categorical_columns,
           columns_with_nulls, columns_with_outliers, constant_columns, correlation_pairs
    FROM audit.profiling_reports
    WHERE run_id = ?
    """,
    [latest_run_id],
).pl()

print(f"Rows in latest run: {latest_profiling.height}")
latest_profiling.head(10)

Latest profiling run_id: raw_2026-07-07T19:16:43
Rows in latest run: 369


run_id,layer,table_name,generated_at,report_version,total_rows,total_columns,numeric_columns,categorical_columns,columns_with_nulls,columns_with_outliers,constant_columns,correlation_pairs
str,str,str,datetime[μs],i32,i64,i32,i32,i32,i32,i32,i32,i32
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures""",2026-07-07 19:15:27,1,7789,15,7,8,1,2,5,21
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_archive""",2026-07-07 19:15:27,1,3094,13,6,7,0,1,3,15
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_archive_areacodes""",2026-07-07 19:15:27,1,121,3,1,2,0,0,0,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_archive_elements""",2026-07-07 19:15:27,1,2,2,1,1,0,0,0,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_archive_flags""",2026-07-07 19:15:27,1,1,2,0,2,0,0,2,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_archive_itemcodes""",2026-07-07 19:15:27,1,1,2,1,1,0,0,2,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_areacodes""",2026-07-07 19:15:27,1,163,3,1,2,0,1,0,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_costcategorys""",2026-07-07 19:15:27,1,1,2,1,1,0,0,2,0
"""raw_2026-07-07T19:16:43""","""raw""","""asti_expenditures_flags""",2026-07-07 19:15:27,1,5,2,0,2,0,0,0,0


### Coverage check — does the latest run cover every raw table?

If any table is missing from the latest run (profiled in an earlier run only), fall
back to including its most recent available report so nothing silently disappears
from the analysis below.

In [16]:
all_profiled_tables = conn.execute(
    "SELECT DISTINCT table_name FROM audit.profiling_reports"
).pl()["table_name"]

missing_from_latest = set(all_profiled_tables) - set(latest_profiling["table_name"])

if missing_from_latest:
    print(f"{len(missing_from_latest)} tables missing from latest run — backfilling from their own latest report")
    backfill = conn.execute(
        f"""
        SELECT run_id, layer, table_name, generated_at, report_version,
               total_rows, total_columns, numeric_columns, categorical_columns,
               columns_with_nulls, columns_with_outliers, constant_columns, correlation_pairs
        FROM audit.profiling_reports
        WHERE table_name IN ({', '.join(f"'{t}'" for t in missing_from_latest)})
        QUALIFY ROW_NUMBER() OVER (PARTITION BY table_name ORDER BY generated_at DESC) = 1
        """
    ).pl()
    latest_profiling = pl.concat([latest_profiling, backfill])
else:
    print("All raw tables are covered by the latest run — no backfill needed.")

print(f"Final table count: {latest_profiling.height}")
assert latest_profiling.height == latest_profiling["table_name"].n_unique(), "Should be one row per table"
assert latest_profiling.height == len(all_profiled_tables), "Should cover every profiled table"
print("OK — one row per table, full coverage confirmed.")

All raw tables are covered by the latest run — no backfill needed.
Final table count: 369
OK — one row per table, full coverage confirmed.


## 2. Overview — how common is each signal?

Before ranking anything, get a feel for scale: how many tables have any constant
columns, any outlier columns, any correlated pairs — and what's the range.

In [17]:
overview = latest_profiling.select(
    pl.col("constant_columns").min().alias("min_constant"),
    pl.col("constant_columns").max().alias("max_constant"),
    (pl.col("constant_columns") > 0).sum().alias("n_tables_with_constant"),
    pl.col("columns_with_outliers").min().alias("min_outliers"),
    pl.col("columns_with_outliers").max().alias("max_outliers"),
    (pl.col("columns_with_outliers") > 0).sum().alias("n_tables_with_outliers"),
    pl.col("correlation_pairs").min().alias("min_corr"),
    pl.col("correlation_pairs").max().alias("max_corr"),
    (pl.col("correlation_pairs") > 0).sum().alias("n_tables_with_corr"),
)

print(f"Total tables: {latest_profiling.height}")
overview

Total tables: 369


min_constant,max_constant,n_tables_with_constant,min_outliers,max_outliers,n_tables_with_outliers,min_corr,max_corr,n_tables_with_corr
i32,i32,u32,i32,i32,u32,i32,i32,u32
0,9,106,0,54,167,0,2485,79


## 3. Constant columns

Ranked by `constant_columns` descending. A constant column (single distinct value
across all rows, e.g. `Cost Category Code` always `300`) carries no information and
is a strong candidate to drop in staging — but confirm via `report_json` before
dropping blind, in case the "constant" value is actually meaningful metadata
(e.g. a source/version tag).

In [18]:
constant_ranked = (
    latest_profiling
    .filter(pl.col("constant_columns") > 0)
    .select(["table_name", "total_rows", "total_columns", "constant_columns"])
    .sort("constant_columns", descending=True)
)

print(f"Tables with at least one constant column: {constant_ranked.height}")
constant_ranked.head(30)

Tables with at least one constant column: 106


table_name,total_rows,total_columns,constant_columns
str,i64,i32,i32
"""asti_researchers""",3800,18,9
"""asti_expenditures""",7789,15,5
"""holidays""",88537,10,5
"""asti_researchers_archive""",3154,13,4
"""population""",168405,13,4
"""asti_expenditures_archive""",3094,13,3
"""climate_change_emissions_indicators_itemcodes""",0,3,3
"""cost_affordability_healthy_diet_coahd_itemcodes""",0,3,3
"""deflators_itemcodes""",0,3,3


## 4. Columns with outliers

Ranked by `columns_with_outliers` descending. Unlike constant columns, this is *not*
automatically a problem — outliers in a `Value` column of a commodity price or trade
volume table might be entirely legitimate (a real price spike, a real large exporter).
Flag for a closer look in 04, don't assume action is needed.

In [19]:
outliers_ranked = (
    latest_profiling
    .filter(pl.col("columns_with_outliers") > 0)
    .select(["table_name", "total_rows", "total_columns", "columns_with_outliers"])
    .sort("columns_with_outliers", descending=True)
)

print(f"Tables with at least one column flagged for outliers: {outliers_ranked.height}")
outliers_ranked.head(30)

Tables with at least one column flagged for outliers: 167


table_name,total_rows,total_columns,columns_with_outliers
str,i64,i32,i32
"""commodity_prices""",798,72,54
"""trade_matrix""",6640547,90,27
"""emdat""",27681,47,17
"""commodity_indices""",798,17,10
"""emissions_totals""",2500090,15,6
"""emissions_land_use_fires""",428963,15,5
"""inputs_pesticides_trade""",190740,12,5
"""climate_change_emissions_indicators""",678370,12,4
"""emissions_crops""",766730,16,4


## 5. Correlated column pairs

Ranked by `correlation_pairs` descending. A high count here usually means the table
has multiple numeric columns that move together — e.g. a `Value` column and its
`Value (normalized)` counterpart, or several unit variants of the same measurement.
Worth checking in `report_json` which specific columns correlate before assuming
any are truly redundant.

In [20]:
correlation_ranked = (
    latest_profiling
    .filter(pl.col("correlation_pairs") > 0)
    .select(["table_name", "total_rows", "total_columns", "numeric_columns", "correlation_pairs"])
    .sort("correlation_pairs", descending=True)
)

print(f"Tables with at least one correlated column pair: {correlation_ranked.height}")
correlation_ranked.head(30)

Tables with at least one correlated column pair: 79


table_name,total_rows,total_columns,numeric_columns,correlation_pairs
str,i64,i32,i32,i32
"""commodity_prices""",798,72,71,2485
"""trade_matrix""",6640547,90,31,465
"""emdat""",27681,47,22,231
"""commodity_indices""",798,17,16,120
"""asti_researchers""",3800,18,8,28
"""development_assistance_to_agriculture""",13020275,18,8,28
"""employment_indicators_agriculture""",256389,17,8,28
"""employment_indicators_rural""",113187,17,8,28
"""suite_of_gender_indicators""",315014,24,8,28


## 6. Drill into `report_json` for the top-flagged tables

The summary columns tell us *how many* columns are constant/outlier/correlated, but
not *which* ones. Pull `report_json` for the top few tables from sections 4-6 and
extract the actual column names — this is what turns into concrete staging decisions
in 05, so it's worth doing here rather than re-deriving it later.

In [21]:
# Tables to drill into: top 5 from each signal, deduped
drilldown_tables = list(set(
    constant_ranked.head(5)["table_name"].to_list()
    + correlation_ranked.head(5)["table_name"].to_list()
))

print(f"Drilling into {len(drilldown_tables)} tables: {drilldown_tables}")

report_json_df = conn.execute(f"""
    SELECT table_name, report_json
    FROM audit.profiling_reports
    WHERE table_name IN ({', '.join(f"'{t}'" for t in drilldown_tables)})
""").pl()

# In case of duplicate runs, keep only the row matching latest_profiling's chosen run_id
report_json_df = report_json_df.join(
    latest_profiling.select(["table_name", "run_id"]), on="table_name", how="inner"
) if "run_id" in report_json_df.columns else report_json_df

print(f"Fetched report_json for {report_json_df.height} tables")

Drilling into 9 tables: ['asti_researchers_archive', 'asti_researchers', 'emdat', 'trade_matrix', 'commodity_prices', 'population', 'holidays', 'asti_expenditures', 'commodity_indices']
Fetched report_json for 9 tables


In [22]:
def find_constant_columns(report_json_str: str) -> list[str]:
    """A column is constant if unique_counts.distinct_count == 1 (ignoring all-null columns,
    which are a null issue tracked separately, not a constant-value issue)."""
    report = json.loads(report_json_str)
    unique_counts = report.get("unique_counts", {})
    null_counts = report.get("null_counts", {}).get("columns", {})
    constants = []
    for col, info in unique_counts.items():
        distinct = info.get("distinct_count")
        null_pct = null_counts.get(col, {}).get("null_pct", 0)
        if distinct == 1 and null_pct < 100:
            constants.append(col)
    return constants

def find_all_null_columns(report_json_str: str) -> list[str]:
    """Separately surface 100%-null columns — not 'constant' in the profiling report's
    sense, but equally dead weight for staging."""
    report = json.loads(report_json_str)
    null_counts = report.get("null_counts", {}).get("columns", {})
    return [col for col, info in null_counts.items() if info.get("null_pct") == 100.0]

def count_wide_year_columns(report_json_str: str) -> int:
    """Detect FAOSTAT-style wide year columns (e.g. Y1986, Y1986F, Y2024, Y2024F).
    A table with many of these is almost certainly pivoted-wide by year and should be
    unpivoted to long format (one row per year) in staging — that alone explains most
    of a table's null-column and correlation-pair counts without those columns being
    a real quality problem."""
    report = json.loads(report_json_str)
    schema = report.get("schema", {})
    import re
    year_col_pattern = re.compile(r"^Y\d{4}F?$")
    return sum(1 for col in schema if year_col_pattern.match(col))

drilldown_results = report_json_df.with_columns(
    pl.col("report_json").map_elements(find_constant_columns, return_dtype=pl.List(pl.Utf8)).alias("constant_column_names"),
    pl.col("report_json").map_elements(find_all_null_columns, return_dtype=pl.List(pl.Utf8)).alias("all_null_column_names"),
    pl.col("report_json").map_elements(count_wide_year_columns, return_dtype=pl.Int64).alias("wide_year_column_count"),
).select(["table_name", "constant_column_names", "all_null_column_names", "wide_year_column_count"])

drilldown_results

table_name,constant_column_names,all_null_column_names,wide_year_column_count
str,list[str],list[str],i64
"""asti_expenditures""","[""Cost Category Code"", ""Cost Category"", … ""Institution""]","[""Note""]",0
"""asti_researchers""","[""Degree Code"", ""Degree"", … ""Unit""]","[""Note""]",0
"""asti_researchers_archive""","[""Item Code"", ""Item"", … ""Flag""]",[],0
"""commodity_indices""",[],[],0
"""commodity_prices""",[],[],0
"""emdat""",[],[],0
"""holidays""","[""types""]","[""fixed"", ""global"", … ""launch_year""]",0
"""population""","[""Item Code"", ""Item"", ""Unit""]","[""Note""]",0
"""trade_matrix""",[],[],78


## 7. Full-dataset scan: which tables are wide-format by year?

`trade_matrix` turned out to have 78 wide year columns (`Y1986`...`Y2024F`), which fully
explained its null-column and correlation-pair counts — not a quality problem, but a
structural one requiring an unpivot in staging. The drilldown above only checked 9 tables;
this section checks all 369, using `report_json.schema` directly, to find every table with
this pattern so 05's cleaning plan covers all of them, not just `trade_matrix`.

In [23]:
import re

year_col_pattern = re.compile(r"^Y\d{4}F?$")

def count_wide_year_columns_from_json(report_json_str: str) -> int:
    report = json.loads(report_json_str)
    schema = report.get("schema", {})
    return sum(1 for col in schema if year_col_pattern.match(col))

# Pull report_json for every table in the latest run (plus any backfilled tables) — larger
# fetch than the earlier drilldown, but only run once, and only extracts one integer per table
all_report_json = conn.execute(
    f"""
    SELECT table_name, run_id, report_json
    FROM audit.profiling_reports
    WHERE table_name IN ({', '.join(f"'{t}'" for t in latest_profiling['table_name'].to_list())})
    """
).pl()

# Keep only each table's chosen run (matches latest_profiling's run_id per table,
# including any backfilled rows from section 1)
all_report_json = all_report_json.join(
    latest_profiling.select(["table_name", "run_id"]),
    on=["table_name", "run_id"],
    how="inner",
)

wide_year_full = all_report_json.with_columns(
    pl.col("report_json").map_elements(count_wide_year_columns_from_json, return_dtype=pl.Int64).alias("wide_year_column_count")
).select(["table_name", "wide_year_column_count"])

wide_year_ranked = wide_year_full.filter(pl.col("wide_year_column_count") > 0).sort("wide_year_column_count", descending=True)

print(f"Tables with any wide year columns: {wide_year_ranked.height} / {wide_year_full.height}")
wide_year_ranked

Tables with any wide year columns: 1 / 369


table_name,wide_year_column_count
str,i64
"""trade_matrix""",78


## 8. Build the shortlist

Combine all four signals into a single table: any table with a constant column, an
outlier column, a correlated pair, or wide year columns gets flagged with the specific
reason(s). Tables flagged only for `wide_year_columns` are a *structural* note (needs
unpivoting in staging), not a quality problem — kept separate in `flagged_for` so 05
can tell the two apart.

Two categories of likely-noise are split out separately below, rather than left mixed
into the actionable list:

1. **Trivial lookup tables** (`total_rows <= 1`) — flagged `constant_columns` in every
   column by definition, not actionable.
2. **Small lookup tables flagged only for a single outlier column** (`total_columns <= 3`,
   `columns_with_outliers == 1`, no other reason) — this pattern repeats across ~90
   `_areacodes`/`_itemcodes` tables and is **confirmed** (see the verification cell and
   closing notes below) to be a numeric-code-range artifact: individual country codes
   clustered low, mixed with a handful of high-valued regional/world aggregate codes in
   the same column. Real FAOSTAT convention, not a quality issue.

Carry `profiling_shortlist` into 04 for a closer look at actual values, and into 05 for
concrete staging decisions.

In [24]:
flagged_constant = set(constant_ranked["table_name"].to_list())
flagged_outliers = set(outliers_ranked["table_name"].to_list())
flagged_corr = set(correlation_ranked["table_name"].to_list())
flagged_wide_year = set(wide_year_ranked["table_name"].to_list())

all_flagged = flagged_constant | flagged_outliers | flagged_corr | flagged_wide_year

def reasons_for(table_name: str) -> str:
    reasons = []
    if table_name in flagged_constant:
        reasons.append("constant_columns")
    if table_name in flagged_outliers:
        reasons.append("outlier_columns")
    if table_name in flagged_corr:
        reasons.append("correlated_pairs")
    if table_name in flagged_wide_year:
        reasons.append("wide_year_columns")
    return ", ".join(reasons)

flagged_full = (
    latest_profiling
    .filter(pl.col("table_name").is_in(all_flagged))
    .select(["table_name", "total_rows", "total_columns", "constant_columns", "columns_with_outliers", "correlation_pairs"])
    .join(wide_year_full, on="table_name", how="left")
    .with_columns(
        pl.col("table_name").map_elements(reasons_for, return_dtype=pl.Utf8).alias("flagged_for")
    )
    .sort("table_name")
)

# 1. Trivial 1-row lookup tables — constant by construction, not actionable.
trivial_lookup_tables = flagged_full.filter(pl.col("total_rows") <= 1)
remaining = flagged_full.filter(pl.col("total_rows") > 1)

# 2. Small lookup tables (<=3 columns) flagged ONLY for a single outlier column.
#    Suspected code-range artifact, NOT yet confirmed — verify in 04 before treating
#    as resolved. Kept separate so it doesn't silently disappear from view.
suspected_lookup_code_range = remaining.filter(
    (pl.col("total_columns") <= 3)
    & (pl.col("columns_with_outliers") == 1)
    & (pl.col("flagged_for") == "outlier_columns")
)
profiling_shortlist = remaining.filter(
    ~(
        (pl.col("total_columns") <= 3)
        & (pl.col("columns_with_outliers") == 1)
        & (pl.col("flagged_for") == "outlier_columns")
    )
)

print(f"Total tables flagged (raw): {flagged_full.height} / {latest_profiling.height}")
print(f"Of which trivial 1-row lookup tables (excluded): {trivial_lookup_tables.height}")
print(f"Of which confirmed lookup code-range artifact (excluded, see notes): {suspected_lookup_code_range.height}")
print(f"Actionable shortlist: {profiling_shortlist.height}")
print(f"Of which flagged for wide_year_columns: {len(flagged_wide_year)}")
profiling_shortlist

Total tables flagged (raw): 225 / 369
Of which trivial 1-row lookup tables (excluded): 45
Of which confirmed lookup code-range artifact (excluded, see notes): 90
Actionable shortlist: 90
Of which flagged for wide_year_columns: 1


table_name,total_rows,total_columns,constant_columns,columns_with_outliers,correlation_pairs,wide_year_column_count,flagged_for
str,i64,i32,i32,i32,i32,i64,str
"""asti_expenditures""",7789,15,5,2,21,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""asti_expenditures_archive""",3094,13,3,1,15,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""asti_researchers""",3800,18,9,1,28,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""asti_researchers_archive""",3154,13,4,1,15,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""climate_change_emissions_indicators""",678370,12,1,4,15,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""commodity_indices""",798,17,0,10,120,0,"""outlier_columns, correlated_pairs"""
"""commodity_prices""",798,72,0,54,2485,0,"""outlier_columns, correlated_pairs"""
"""commoditybalances_non_food""",1184986,13,1,3,15,0,"""constant_columns, outlier_columns, correlated_pairs"""
"""commoditybalances_non_food_2010""",127558,14,1,2,15,0,"""constant_columns, outlier_columns, correlated_pairs"""


### Verify the lookup code-range hypothesis on one example

Pull `report_json` for one table from `suspected_lookup_code_range` and check the
actual statistics on its flagged column, rather than trusting the guess. If the
"outlier" is a high-value aggregate code (e.g. a regional/world code far above the
range of individual country codes), the hypothesis holds and this exclusion is safe
to keep. If not, pull `suspected_lookup_code_range` back into the real shortlist.

In [25]:
example_table = suspected_lookup_code_range["table_name"][0]
print(f"Checking: {example_table}")

example_json = conn.execute(
    """
    SELECT report_json FROM audit.profiling_reports
    WHERE table_name = ?
    ORDER BY generated_at DESC
    LIMIT 1
    """,
    [example_table],
).fetchone()[0]

example_report = json.loads(example_json)
print("Schema:", example_report.get("schema"))
print("Statistics:", example_report.get("statistics"))

Checking: asti_expenditures_areacodes
Schema: {'Area Code': 'BIGINT', 'M49 Code': 'VARCHAR', 'Area': 'VARCHAR'}
Statistics: {'Area Code': {'min': 3, 'max': 5817, 'mean': 1255.717791411043, 'median': 167.0, 'stddev': 2161.547792240158}}


### Notes — Data Profiling

- *(fill in after reviewing the shortlist and drilldown above)*
- From section 6: which constant columns are truly dead weight vs. meaningful metadata
  worth keeping (e.g. a `Cost Category` that's constant *within this table* but varies
  across the domain as a whole)?
- Any all-null columns (section 6's `all_null_column_names`) — these are distinct from
  constant columns and equally safe to drop.
- **Resolved — `trade_matrix`'s 465 correlation pairs and 78 null columns:** not a data
  quality problem. `report_json` shows `trade_matrix` is wide-format with one column pair
  per year (`Y1986`/`Y1986F` ... `Y2024`/`Y2024F`, 78 columns for 39 years). Adjacent years
  are naturally correlated, and most year/country/item combos are null simply because that
  trade didn't happen that year.
- **Resolved — section 7 full-dataset scan:** only **1 of 369 tables** (`trade_matrix`
  itself) has any wide year columns. Not a systemic FAOSTAT pattern — a genuine one-off.
  **Action for 05:** unpivot `trade_matrix` specifically to long format (one row per year)
  in staging; no shared macro needed elsewhere.
- **Resolved — trivial lookup tables:** the first full shortlist pass (225/369 tables) was
  dominated by 1-row `_flags`/`_itemcodes`/`_costcategorys`-style lookup tables, which are
  flagged `constant_columns` in every column by construction — not actionable. These are now
  split into `trivial_lookup_tables` and excluded from `profiling_shortlist`.
- **Resolved — lookup code-range artifact confirmed:** verification on `asti_expenditures_areacodes`
  shows `Area Code` ranges 3-5817, but median is only 167 while mean is 1255.7 — a strongly
  right-skewed distribution with stddev (2161) larger than the mean. This confirms individual
  country codes (clustered low, ~100s) are mixed with a small number of high-valued
  regional/aggregate codes (e.g. "World", continents) in the same column. Real FAOSTAT
  convention, not a data quality issue. **Action for 05:** none needed — these ~90
  `_areacodes`/`_itemcodes` tables can be excluded from cleaning attention; if aggregate rows
  ever need excluding for a specific analysis, filter by code range at the staging/mart layer,
  not by fixing the raw lookup tables.
- **Follow-up for 04:** `commodity_prices` has 2485 correlation pairs (higher than
  `trade_matrix`'s 465) across 72 columns, but `wide_year_column_count = 0` — the wide-year
  explanation doesn't apply here. Worth checking actual column names in 04: could be wide by
  commodity/price-series instead of year, or genuinely redundant unit-conversion columns.
- Cross-reference `profiling_shortlist` against 02's `shortlist` — tables appearing in
  both need cleaning **and** structural attention.
- Carry both into **04 — Raw Data Exploration** for a closer look at actual values, and
  into **05 — Data Cleaning Plan** for concrete staging decisions.

In [26]:
# Close the connection
conn.close()
print("Connection closed")

Connection closed
